# Chat Templates and Data Preparation — In-Depth Guide

This notebook is built on facts **verified live on a real server** (see `smoke/verify_templates.py`). No guesses.

## Why This Matters

Chat templates and data formats are the **quietest failure mode** in fine-tuning:
- Wrong marker → masking silently breaks, model degrades
- Wrong format → loss is computed but is meaningless
- Double `<bos>` → corrupted model output
- `tools=` argument is silently ignored (Gemma 4!)

Nothing throws an error — training just turns out bad. That's why understanding this in depth is essential.

## Table of Contents

1. **What Is a Chat Template?** — Jinja → string mechanics
2. **Side-by-side Render** — 5 templates compared
3. **`add_generation_prompt`** — what gets appended for generation?
4. **`tools=` Parameter** — which templates support it?
5. **`enable_thinking=`** — actual behavior of thinking templates
6. **Multi-turn + System Prompt**
7. **Data Formats** — Alpaca / ShareGPT / OpenAI
8. **`standardize_data_formats`** — what does it actually do?
9. **`formatting_prompts_func` Patterns**
10. **Vision Data Format**
11. **Tool Calling Data Format**
12. **Masking Verification** — `train_on_responses_only`
13. **Common Pitfalls**

## 1. What Is a Chat Template?

A chat template is a **Jinja** template that turns a list of messages (`[{role, content}, ...]`) into a single **string**.

### Three Places It Lives
1. **Inside the tokenizer itself** — the `chat_template` field in `tokenizer_config.json` (HF default)
2. **Unsloth `CHAT_TEMPLATES` dict** — 47 hard-coded templates in `unsloth/chat_templates.py`
3. **`get_chat_template(tokenizer, name)`** — picks one from the dict above and injects it into the tokenizer

### Why Use Unsloth's Own Template?
- The template in the HF tokenizer is sometimes incomplete or wrong
- Unsloth's version is tested and aligned with the masking logic
- Optimized per model family (e.g. Gemma 4's `<bos>` quirk)

In [ ]:
import unsloth                                # MUST be the very first import
from unsloth.chat_templates import (
    get_chat_template,
    standardize_data_formats,
    train_on_responses_only,
    CHAT_TEMPLATES,                           # raw dict
)
from transformers import AutoTokenizer
from datasets import Dataset

# Available templates
print(f'{len(CHAT_TEMPLATES)} templates registered')
print(sorted(CHAT_TEMPLATES.keys()))

### Prepare the Tokenizers

We'll compare 5 models. Only loading tokenizers — no models. Fast.

In [ ]:
TOK_FOR = {
    'qwen3-instruct':   'unsloth/Qwen3-4B-Instruct-2507',
    'qwen3-thinking':   'unsloth/Qwen3-4B-Thinking-2507',
    'gemma-4':          'unsloth/gemma-4-E2B-it',
    'gemma-4-thinking': 'unsloth/gemma-4-E2B-it',     # same tokenizer, different template
    'llama-3.1':        'unsloth/Llama-3.1-8B-Instruct',
}

tokenizers = {}
for name, path in TOK_FOR.items():
    tok = AutoTokenizer.from_pretrained(path)
    tok = get_chat_template(tok, chat_template=name)
    tokenizers[name] = tok
    print(f'OK: {name}')

## 2. Side-by-side Render

**The same conversation**, 5 different templates — each output is distinct. These differences are mandatory: every model was trained with its own marker set.

In [ ]:
msgs = [
    {'role': 'user',      'content': 'What is interest?'},
    {'role': 'assistant', 'content': 'Interest is the time value of money.'},
]

for name, tok in tokenizers.items():
    print('=' * 60)
    print(name)
    print('=' * 60)
    print(tok.apply_chat_template(msgs, tokenize=False))
    print()

### Marker Comparison (verified)

| Template | Open / Close | Assistant role | BOS |
|---|---|---|---|
| `qwen3-instruct` | `<|im_start|>` / `<|im_end|>` (symmetric) | `assistant` | none |
| `qwen3-thinking` | `<|im_start|>` / `<|im_end|>` | `assistant` + auto `<think>` | none |
| `gemma-4` | **`<|turn>` / `<turn|>` (asymmetric!)** | **`model`** | **`<bos>`** |
| `gemma-4-thinking` | `<|turn>` / `<turn|>` | `model` + `<|channel>thought` | `<bos>` |
| `llama-3.1` | `<|start_header_id|>` / `<|eot_id|>` | `assistant` (injects a system message!) | `<|begin_of_text|>` |

### Observations
- **Gemma 4 is asymmetric** — opening is `<|turn>` but closing is `<turn|>`. Get it wrong and masking breaks.
- **Gemma 4's assistant role is `model`** (not `assistant`!)
- **Llama 3.1 auto-injects a system prompt** — things like "Cutting Knowledge Date"
- **Qwen3-thinking auto-opens a `<think>` block**

## 3. `add_generation_prompt=True`

Required at inference time. Appends the **assistant header** to the end of the template so the model knows to write a reply. **Not needed at training time** (it's already in the data).

**Important:** every template appends something different. Here is the actual output (verified).

In [ ]:
user_only = [{'role': 'user', 'content': 'What is interest?'}]

for name, tok in tokenizers.items():
    no_gen   = tok.apply_chat_template(user_only, tokenize=False, add_generation_prompt=False)
    with_gen = tok.apply_chat_template(user_only, tokenize=False, add_generation_prompt=True)
    suffix   = with_gen[len(no_gen):]
    print(f'{name:20} appended = {suffix!r}')

### What Gets Appended by Generation Prompt (verified)

| Template | Appended |
|---|---|
| `qwen3-instruct` | `<|im_start|>assistant\n` |
| `qwen3-thinking` | **`<|im_start|>assistant\n<think>\n`** (auto-thinking!) |
| `gemma-4` | `<|turn>model\n` |
| `gemma-4-thinking` | **`<|turn>model\n<|channel>thought\n<channel|>`** (custom thinking marker!) |
| `llama-3.1` | `<|start_header_id|>assistant<|end_header_id|>\n\n` |

### Takeaways
- **Thinking models** kick off the generation prompt with thinking themselves
- That's why the model starts reasoning automatically — you don't need to write `<think>`
- Even if you pass `enable_thinking=False` at inference time, most thinking templates ignore it (see Section 5)

## 4. `tools=` Parameter — CRITICAL

Tool calling (function calling) requires a `tools` branch in the template. **Without it, the `tools=` argument is silently ignored** — no error, nothing.

Let's test:

In [ ]:
tools = [{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': 'Get weather for a city',
        'parameters': {
            'type': 'object',
            'properties': {'city': {'type': 'string'}},
            'required': ['city'],
        },
    },
}]
test_msgs = [{'role': 'user', 'content': 'What is the weather in Istanbul?'}]

for name, tok in tokenizers.items():
    without = tok.apply_chat_template(test_msgs, tokenize=False)
    with_t  = tok.apply_chat_template(test_msgs, tokenize=False, tools=tools)
    supported = with_t != without
    diff = len(with_t) - len(without)
    print(f'{name:20} tools= supported = {supported}  (diff: +{diff} chars)')

### Tools Support (verified)

| Template | `tools=` Support | Note |
|---|---|---|
| `qwen3-instruct` | ✅ +600 chars | Hermes-style `<tool_call>` |
| `qwen3-thinking` | ✅ +600 chars | Same, with thinking |
| **`gemma-4`** | ❌ **0 chars** | **Silently ignored!** |
| **`gemma-4-thinking`** | ❌ **0 chars** | **Silently ignored!** |
| `llama-3.1` | ✅ +672 chars | JSON function format |

### Practical Conclusion
- **Use Qwen3 or Llama 3.1 for tool-calling SFT**
- If you want tool calling with Gemma 4: embed the JSON schema manually in the system prompt and train a custom `<tool_call>` marker
- Don't be fooled: `apply_chat_template(messages, tools=[...])` doesn't error on Gemma 4 — it just does nothing

## 5. `enable_thinking=` Parameter

For thinking models. Behavior is template-specific and **surprising**.

In [ ]:
for name in ['qwen3-instruct', 'qwen3-thinking', 'gemma-4', 'gemma-4-thinking']:
    tok = tokenizers[name]
    print(f'\n--- {name} ---')
    for et in [False, True]:
        try:
            out = tok.apply_chat_template(
                [{'role': 'user', 'content': 'What is interest?'}],
                tokenize=False, add_generation_prompt=True,
                enable_thinking=et,
            )
            tail = out[-80:]
            print(f'  enable_thinking={et}: ...{tail!r}')
        except Exception as e:
            print(f'  enable_thinking={et}: FAIL {e}')

### `enable_thinking` Behavior (verified)

| Template | False | True |
|---|---|---|
| `qwen3-instruct` | ignored | ignored |
| `qwen3-thinking` | **ignored** — always appends `<think>\n` | same |
| `gemma-4` | normal output | **prepends `<|think|>\n<turn|>`!** |
| `gemma-4-thinking` | uses **default thinking marker** (`<|channel>thought`) | uses **alternative marker** (`<|think|>`) |

### Key Points
- `qwen3-thinking` + `enable_thinking=False` → **doesn't work**, the model still starts with `<think>`
- To turn thinking off either **change the chat template** (use `qwen3-instruct`) or post-process and strip the `<think>...</think>` block
- Even `gemma-4` (non-thinking) can be coerced into thinking-style output with `enable_thinking=True`!

## 6. Multi-turn + System Prompt

In [ ]:
multi = [
    {'role': 'system',    'content': 'You are a helpful Turkish assistant.'},
    {'role': 'user',      'content': 'What is interest?'},
    {'role': 'assistant', 'content': 'Interest is the time value of money.'},
    {'role': 'user',      'content': 'Give an example.'},
]

for name in ['qwen3-instruct', 'gemma-4']:
    print('=' * 60)
    print(name)
    print('=' * 60)
    print(tokenizers[name].apply_chat_template(multi, tokenize=False, add_generation_prompt=True))
    print()

## 7. Data Formats

There are three main formats. Each must be converted into the `[{role, content}]` shape that `apply_chat_template` expects:

### 7a. Alpaca (Sebastian Raschka ch07, legacy format)
```python
{'instruction': 'Define interest', 'input': '', 'output': 'Interest is the time value of money.'}
```
Convert manually → `{role: 'user', content: instruction + (input if any)}` + `{role: 'assistant', content: output}`

### 7b. ShareGPT (legacy Alpaca-style datasets)
```python
{'conversations': [
    {'from': 'human', 'value': 'Define interest'},
    {'from': 'gpt',   'value': 'Interest is the time value of money.'},
]}
```
→ `standardize_data_formats` converts this automatically

### 7c. OpenAI messages (modern, preferred)
```python
{'messages': [
    {'role': 'user',      'content': 'Define interest'},
    {'role': 'assistant', 'content': 'Interest is the time value of money.'},
]}
```
→ Already correct; no conversion needed

## 8. `standardize_data_formats` — Actual Behavior

**According to the source code:** it only kicks in when there's a `conversations` column. Otherwise it returns the dataset unchanged.

Aliases:
- system: `['system']`
- user: `['user', 'human', 'input']`
- assistant: `['gpt', 'assistant', 'output']`

Now let's test all 3 formats:

In [ ]:
# 8a. Alpaca → unchanged (no 'conversations' col)
alpaca = Dataset.from_list([
    {'instruction': 'Define interest', 'input': '', 'output': 'Interest is the time value of money.'},
    {'instruction': 'Translate',       'input': 'hello', 'output': 'merhaba'},
])
print('--- Alpaca ---')
print(f'BEFORE cols: {alpaca.column_names}')
out = standardize_data_formats(alpaca)
print(f'AFTER  cols: {out.column_names}')
print(f'Sample[0]:   {out[0]}')
print('=> Alpaca UNCHANGED (manual conversion required)')

In [ ]:
# 8b. ShareGPT → converted (needs at least 2 rows for role detection)
sharegpt = Dataset.from_list([
    {'conversations': [
        {'from': 'human', 'value': 'Define interest'},
        {'from': 'gpt',   'value': 'Interest is the time value of money.'},
    ]},
    {'conversations': [
        {'from': 'human', 'value': 'Translate hello'},
        {'from': 'gpt',   'value': 'merhaba'},
    ]},
])
print('--- ShareGPT ---')
print(f'BEFORE: {sharegpt[0]}')
out = standardize_data_formats(sharegpt)
print(f'AFTER:  {out[0]}')
print('=> ShareGPT (from/value/human/gpt) → OpenAI (role/content/user/assistant)')

In [ ]:
# 8c. OpenAI messages → unchanged (no 'conversations' col, function exits early)
openai = Dataset.from_list([
    {'messages': [
        {'role': 'user',      'content': 'Define interest'},
        {'role': 'assistant', 'content': 'Interest is the time value of money.'},
    ]},
])
print('--- OpenAI ---')
print(f'BEFORE cols: {openai.column_names}')
out = standardize_data_formats(openai)
print(f'AFTER  cols: {out.column_names}')
print('=> OpenAI UNCHANGED (already in the correct format)')

### Practical Rule

| Source | `standardize_data_formats` | Pipeline |
|---|---|---|
| Alpaca | does nothing | **convert manually** → `conversations: [{role, content}]` |
| ShareGPT | `{from, value}` → `{role, content}` | call directly, works |
| OpenAI messages | does nothing | **rename the column to `conversations`** or use `messages` |

## 9. `formatting_prompts_func` Patterns

If you set `SFTConfig(dataset_text_field='text')`, the dataset must contain a **'text' column**. We build that column with `formatting_prompts_func`.

In [ ]:
# 9a. Qwen3 — standard pattern
tok = tokenizers['qwen3-instruct']

def fmt_qwen3(examples):
    return {'text': [
        tok.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        for c in examples['conversations']
    ]}

# Test
data = Dataset.from_list([
    {'conversations': [
        {'role': 'user', 'content': 'What is interest?'},
        {'role': 'assistant', 'content': 'Interest is the time value of money.'},
    ]},
])
out = data.map(fmt_qwen3, batched=True)
print(out[0]['text'])

In [ ]:
# 9b. Gemma 4 — '<bos>' STRIP is mandatory
# Double bos → broken model. The processor will add it itself.
tok = tokenizers['gemma-4']

def fmt_gemma4(examples):
    return {'text': [
        tok.apply_chat_template(c, tokenize=False, add_generation_prompt=False).removeprefix('<bos>')
        for c in examples['conversations']
    ]}

out = data.map(fmt_gemma4, batched=True)
print('After Gemma 4 format (no bos at start):')
print(repr(out[0]['text'][:100]))

## 10. Vision Data Format

In vision SFT, `content` is **not a string** — it's a list of typed content blocks.

In [ ]:
# Vision conversation — image + text mixed
vision_msg = {
    'messages': [
        {'role': 'user', 'content': [
            {'type': 'text',  'text': 'What is the LaTeX in this image?'},
            {'type': 'image', 'image': 'https://example.com/img.png'},  # must be a real PIL.Image
        ]},
        {'role': 'assistant', 'content': [
            {'type': 'text', 'text': r'\frac{a}{b}'},
        ]},
    ]
}
print('Vision format:')
import json
print(json.dumps(vision_msg, indent=2, ensure_ascii=False))

### Vision Pipeline Notes
1. **Don't use** `Dataset.map(fmt_func)` — Datasets can't pickle PIL.Image. Use a list comprehension:
   ```python
   converted = [convert_to_conversation(s) for s in dataset]
   ```
2. Three SFTConfig settings are **mandatory**:
   ```python
   remove_unused_columns = False,
   dataset_text_field = '',
   dataset_kwargs = {'skip_prepare_dataset': True},
   ```
3. Data collator: `UnslothVisionDataCollator(model, tokenizer)`

## 11. Tool Calling Data Format

As we saw in Section 4, only `qwen3-*` and `llama-3.1` support `tools=`. The training data is shaped accordingly.

In [ ]:
# Tool calling SFT — multi-turn assistant→tool→assistant
tool_conv = [
    {'role': 'system', 'content': 'You are a weather assistant.'},
    {'role': 'user', 'content': 'What is the weather in Istanbul?'},
    {'role': 'assistant', 'content': '',
     'tool_calls': [{
         'id': 'call_1', 'type': 'function',
         'function': {'name': 'get_weather', 'arguments': {'city': 'Istanbul'}}
     }]},
    {'role': 'tool', 'tool_call_id': 'call_1', 'name': 'get_weather',
     'content': '{"city": "Istanbul", "temp_c": 18, "condition": "cloudy"}'},
    {'role': 'assistant', 'content': 'It is currently 18°C and cloudy in Istanbul.'},
]

tools = [{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': 'Get weather for a city',
        'parameters': {
            'type': 'object',
            'properties': {'city': {'type': 'string'}},
            'required': ['city'],
        },
    },
}]

tok = tokenizers['qwen3-instruct']
rendered = tok.apply_chat_template(tool_conv, tokenize=False, tools=tools)
print(rendered)

## 12. Masking Verification — `train_on_responses_only`

This function ensures **loss is only computed on assistant responses** during training. Wrong markers → masking silently breaks and loss is computed across the full sequence (bad).

The markers are **model-specific:**

| Model | instruction_part | response_part |
|---|---|---|
| Qwen3 (instruct/thinking) | `'<|im_start|>user\n'` | `'<|im_start|>assistant\n'` |
| Gemma 4 | `'<|turn>user\n'` | **`'<|turn>model\n'`** (not assistant!) |
| Gemma 3 / FunctionGemma | `'<start_of_turn>user\n'` | `'<start_of_turn>model\n'` |
| Llama 3 / 3.1 | `'<|start_header_id|>user<|end_header_id|>\n\n'` | `'<|start_header_id|>assistant<|end_header_id|>\n\n'` |

### Verification Pattern

```python
# After training setup, with the trainer ready:
trainer = train_on_responses_only(
    trainer,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n',
)

# Verify — labels[100] should only show the assistant response
sample = trainer.train_dataset[100]
print('FULL INPUT:')
print(tokenizer.decode(sample['input_ids']))
print('\nUNMASKED LABELS (should only show the response):')
print(tokenizer.decode([
    tokenizer.pad_token_id if x == -100 else x
    for x in sample['labels']
]))
```

If the **user input also** appears in the unmasked labels → **the marker is wrong**. Check it again.

## 13. Common Pitfalls

| # | Pitfall | Result | Fix |
|---|---|---|---|
| 1 | Not stripping `<bos>` for Gemma 4 | Double bos, broken model | `removeprefix('<bos>')` |
| 2 | Using `assistant` marker on Gemma 4 | Masking doesn't work | Use `<|turn>model\n` |
| 3 | Passing `tools=` to Gemma 4 | Silently ignored | Use Qwen3/Llama-3.1, or inject the schema manually in the system prompt |
| 4 | `Dataset.map` on vision data | PIL pickle error | Use a list comprehension |
| 5 | `dataset_text_field='text'` on vision | Data collator format error | `dataset_text_field=''` + `skip_prepare_dataset=True` |
| 6 | Qwen3-thinking + `enable_thinking=False` | Ignored — `<think>` still appears | Switch templates or post-process |
| 7 | `add_generation_prompt=True` at training time | Doubled assistant header | **False** at training time, True at inference |
| 8 | Single-row ShareGPT | Role detection breaks | At least 2 rows with different roles |
| 9 | Passing Alpaca to `standardize_data_formats` | Nothing happens | Convert manually |
| 10 | Not decoding `trainer.train_dataset[100]['labels']` | Wrong masking slips through silently | Always decode-and-print |

## Summary

1. **Chat template = Jinja → string** conversion; every model has its own markers
2. **Gemma 4 doesn't support `tools=`** — silently ignored
3. **Gemma 4's assistant role is `model`** (Qwen3 = `assistant`)
4. **Stripping `<bos>` is mandatory on Gemma 4** (double bos = broken model)
5. **`standardize_data_formats` only converts ShareGPT** — leaves Alpaca / OpenAI alone
6. **Vision needs the mandatory trio:** `remove_unused_columns=False, dataset_text_field='', dataset_kwargs={'skip_prepare_dataset': True}`
7. **Use Qwen3 or Llama 3.1 for tool-calling SFT**
8. **Always verify masking with decode-and-print** — silent failure is very common

**Tested:** `smoke/verify_templates.py` — every claim in this notebook was verified on a live server.